[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/04_cell_specific_six_sweep_fitting.ipynb)

## Colab setup and Step 04 run controls

Run the first setup cell before any imports. In Google Colab it clones this repository, installs `requirements.txt` (including Optuna), changes into the repository root, and puts the repo on `sys.path` so `src.*` imports work.

### Reviewer-facing target scope

By default, Step 04 uses the full available ATF cell set and six current sweeps per cell:

- `ASTROMODEL_STEP04_SELECTED_FILE_IDS=all` -> `selected_file_ids=None`.
- `ASTROMODEL_STEP04_MAX_CELLS=all` -> no cell cap.
- `ASTROMODEL_STEP04_FORCE_RERUN=0` allows reuse only when the completed full run matches this notebook's current optimizer/loss configuration.
- `ASTROMODEL_STEP04_FORCE_RERUN=1` reruns the full optimizer path.

Small subsets are not a reviewer-facing default. Use them only for explicit method-development or generalization experiments, and label the output directory accordingly.

Step 04 fits the six current sweeps for each selected cell. There is no separate region selector in this notebook; to run a deliberate development subset, pass comma-separated file IDs through `ASTROMODEL_STEP04_SELECTED_FILE_IDS`. `ASTROMODEL_STEP04_N_FIT_POINTS` controls trace downsampling for speed/accuracy, not which current sweeps are included.

### Optimizer and loss controls

Use these environment variables before the Step 04 run cell:

Default generation preset: `random_acceptance_160_archive`, validated in `outputs/reviewer_synthesis/step04_proposed_generation_variant_benchmark.md`.

- `ASTROMODEL_STEP04_OPTIMIZER_BACKEND`: `optuna_scalar` by default for the random archive; alternatives are `hybrid`, `least_squares`, or `optuna_multi`.
- `ASTROMODEL_STEP04_OPTUNA_OBJECTIVE`: `acceptance_margin` by default.
- `ASTROMODEL_STEP04_OPTUNA_N_TRIALS`: default `160`.
- `ASTROMODEL_STEP04_OPTUNA_SAMPLER`: `random` by default; alternatives include `tpe` or `nsga2`.
- `ASTROMODEL_STEP04_CANDIDATE_TOP_K`: default `160`, so the retained candidate history is the full random Optuna archive.
- `ASTROMODEL_STEP04_N_STARTS`: default `4`, retained for prior/reference starts used by Step 04 scoring.
- `ASTROMODEL_STEP04_MAX_NFEV_ALL6`: default `20`.
- `ASTROMODEL_STEP04_RUN_HOLDOUT`: `1`/`true`/`yes` by default in this notebook to keep leave-one-sweep-out validation in the reviewer-facing Step 04 contract. The benchmark disabled holdout only to isolate generation speed.
- `ASTROMODEL_STEP04_TRACE_LOSS_TYPE`: `COMBINED` (historical/default), `L2`, `L1`, `HUBER`, or `LOG_COSH`.
- `ASTROMODEL_STEP04_FEATURE_SET`: `primary_no_redundant` by default, so redundant features from Step 02 do not dominate the loss.

### Effective-diverse downstream candidate controls

After accepted six-sweep candidates are available, Step 04 derives a downstream candidate set that keeps candidates separated in effective mechanism space (`P_gap_eff`, `gamma_t_eff`, `gamma_s_eff`, `volume_ratio_wa_wo`). This derived step is inexpensive and can be recomputed from `accepted_cell_ensembles.csv` without rerunning the optimizer.

- `ASTROMODEL_STEP04_EFFECTIVE_DIVERSE_CANDIDATES_PER_CELL`: number of downstream effective-diverse candidates per cell, default `5`.
- `ASTROMODEL_STEP04_EFFECTIVE_DIVERSE_SELECTION_STRATEGY`: default `quality_filtered_effective_maximin`.
- `ASTROMODEL_STEP04_EFFECTIVE_DIVERSE_DISTANCE_THRESHOLD`: log-effective distance threshold used in the diversity summary, default `0.5`.

### Canonical outputs

Canonical downstream files are written under `outputs/cell_fits/`, including full candidate history in `cell_fit_candidates.csv`, accepted ensembles in `accepted_cell_ensembles.csv`, effective-diverse downstream candidates in `effective_diverse_cell_ensembles.csv`, held-out screens in `heldout_current_screen.csv`, and a SQLite audit database when the run path generated one.

In [1]:
# Colab / local repository setup
from pathlib import Path
import os, shutil, subprocess, sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

# In Colab, set these before running the notebook if you want Drive persistence:
# os.environ["ASTROMODEL_STEP04_OUTPUT_DIR"] = "/content/drive/MyDrive/astromodel_outputs/step04_full"
# os.environ["ASTROMODEL_STEP04_BACKUP_DIR"] = "/content/drive/MyDrive/astromodel_outputs/step04_backups"
# os.environ["ASTROMODEL_STEP04_RUN_LABEL"] = "least_squares_full_2026_05"

if _running_in_colab():
    project_root = Path("/content") / PROJECT_DIRNAME
    if not project_root.exists():
        try:
            _run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(project_root)])
        except subprocess.CalledProcessError:
            _run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)])
    os.chdir(project_root)
    requirements = project_root / "requirements.txt"
    if requirements.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next((c for c in candidates if (c / "src").is_dir() and (c / "data").is_dir()), current)
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Optional Colab/Drive data hook. If the repository clone does not contain
# data/2_K+ Pumps Data, set ASTROMODEL_DATA_DIR to a mounted directory with
# that ATF data before running this cell.
expected_data_dir = project_root / "data" / "2_K+ Pumps Data"
external_data_dir = os.environ.get("ASTROMODEL_DATA_DIR")
if external_data_dir and not expected_data_dir.exists():
    src_data = Path(external_data_dir).expanduser().resolve()
    expected_data_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        expected_data_dir.symlink_to(src_data, target_is_directory=True)
    except OSError:
        shutil.copytree(src_data, expected_data_dir)
if not expected_data_dir.exists():
    raise FileNotFoundError(
        f"Missing Step 04 ATF data at {expected_data_dir}. In Colab, mount Drive "
        "or upload data, then set ASTROMODEL_DATA_DIR to the directory containing the ATF files."
    )

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")
print(f"data exists={(project_root / 'data').exists()}, src exists={(project_root / 'src').exists()}")

ASTROMODEL_PROJECT_ROOT=/home/xav/code/astromodel_proving
Working directory=/home/xav/code/astromodel_proving
data exists=True, src exists=True


# Step 04 — Cell-specific six-sweep fitting and accepted ensemble construction

This notebook validates Step 04 using the **expected reviewer-facing astrocyte ODE model**.

Scope of this notebook:
- verify that the implemented `src.astro_model.model` matches the expected equations;
- build cell-specific six-sweep fits under that model;
- use Step 02 region-aware thresholds to define accepted ensembles;
- run held-out-sweep screening as part of the reviewer-facing contract.

Claim boundary:
- this notebook creates accepted cell-specific ensembles;
- it does **not** by itself claim biological degeneracy;
- mechanistic decomposition belongs to Step 05;
- predictive robustness beyond held-out sweeps belongs to Step 06.

In [2]:
from pathlib import Path
import json, os, sys, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

from src.astro_model import build_paramdict, model
from src.step04_cell_fits import acceptance_contract_table, load_step02_outputs_or_run

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT = Path(os.environ.get('ASTROMODEL_PROJECT_ROOT', Path.cwd())).resolve()

# Notebook-only audit/report outputs are separate from canonical Step 04 outputs.
NOTEBOOK_OUTPUT_DIR = Path(os.environ.get(
    'ASTROMODEL_STEP04_NOTEBOOK_OUTPUT_DIR',
    PROJECT_ROOT / 'outputs' / 'cell_fits_step04_model_aligned_demo',
)).resolve()
STEP04_OUTPUT_DIR = Path(os.environ.get(
    'ASTROMODEL_STEP04_OUTPUT_DIR',
    PROJECT_ROOT / 'outputs' / 'cell_fits',
)).resolve()
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEP04_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR
PROJECT_ROOT

PosixPath('/home/xav/code/astromodel_proving')

## Model-alignment audit

In [3]:
def reference_model(z, t, paramdict):
    Cm_a  = paramdict["Astrocyte"]["Cm_a"]
    g_kir = paramdict["Astrocyte"]["g_kir"]
    A = paramdict["Astrocyte"]["A"]
    g_k_a = paramdict["Astrocyte"]["g_k_a"]
    gl_a = paramdict["Astrocyte"]["gl_a"]
    w_a = paramdict["Astrocyte"]["w_a"]
    K_a0 = paramdict["Astrocyte"]["K_a0"]
    Sig_a = paramdict["Astrocyte"]["Sig_a"]
    gama_t = paramdict["Astrocyte"]["gama_t"]
    gama_s = paramdict["Astrocyte"]["gama_s"]
    Z_th = paramdict["Astrocyte"]["Z_th"]
    Z_s = paramdict["Astrocyte"]["Z_s"]
    Va_0 = paramdict["Astrocyte"]["Va_0"]
    Va_s = paramdict["Astrocyte"]["Va_s"]
    Va_l = paramdict["Astrocyte"]["Va_l"]
    P_k = paramdict["Astrocyte"]["P_k"]
    d_gap = paramdict["Astrocyte"]["d_gap"]
    F = paramdict["Astrocyte"]["F"]
    R = paramdict["Astrocyte"]["R"]
    T = paramdict["Astrocyte"]["T"]
    K_o0 =paramdict["external"]["K_o0"]
    w_o = paramdict["external"]["w_o"]
    epsilon = paramdict["external"]["epsilon"]
    idx = np.where(paramdict["external"]["K_bath"]["time"]<=t)[0][-1]
    K_bath = paramdict["external"]["K_bath"]["value"][idx]
    switching_function = paramdict["Astrocyte"].get("switching_function", "sigmoid")
    if "epsilon_middle" in paramdict["external"] and idx == 1:
      epsilon = epsilon*paramdict["external"]["epsilon_middle"]
    if "w_o_middle" in paramdict["external"] and idx == 1:
      w_o = w_o*paramdict["external"]["w_o_middle"]
    Va  = z[0]
    DK_a_t = z[1]
    K_a_s = z[2]
    Kg = z[3]
    DK_a = DK_a_t + K_a_s
    K_a  = K_a0 +DK_a
    DK_o_a = -(w_a/w_o)*DK_a_t
    K_o  = K_o0 + DK_o_a + Kg
    K_ratio = K_o / K_a
    if K_ratio <= 0: K_ratio = 1e-8
    E_k_a = 25.7 * np.log(K_ratio)
    I_k_a = g_k_a*(Va - E_k_a)
    I_Kir = g_kir * np.sqrt(np.abs(K_o))*(Va - E_k_a)*(1/(1+np.exp((Va - E_k_a)/19.2)))
    PH_a = 0.04*(Va - Va_s)
    P_kgap = d_gap*P_k
    exp_neg_PH_a = np.exp(-PH_a)
    denominator = -1 + np.exp(-PH_a)
    if denominator == 0: denominator = 1e-8
    I_kgap = P_kgap * F * PH_a * (1 / denominator) * ((K_a * exp_neg_PH_a) - K_a0)
    I_l_a  = gl_a*(Va - Va_l)
    if switching_function == "sigmoid":
        Th_s = DK_a / (1 + np.exp((Z_th - DK_a_t) * Z_s))
    elif switching_function == "tanh":
        Th_s = DK_a * (0.5 * (1 + np.tanh((DK_a_t - Z_th) * Z_s)))
    elif switching_function == "hill":
        n = paramdict["Astrocyte"].get("hill_coefficient", 2)
        K_d = paramdict["Astrocyte"].get("K_d", 1)
        Th_s = DK_a * ((DK_a_t ** n) / (K_d ** n + DK_a_t ** n))
    else:
        raise ValueError(f"Unknown switching function type: {switching_function}")
    dVa   = (-1.0/Cm_a)*(I_Kir + I_k_a +I_l_a +I_kgap)
    dDK_a_t = -(gama_t*Sig_a/(w_a*F))*(I_Kir + I_k_a)
    dK_a_s = -Th_s*(gama_s*Sig_a/(w_a*F))* I_kgap
    dKg   =  epsilon*(K_bath-K_o)
    return np.asarray([dVa,dDK_a_t,dK_a_s,dKg], dtype=float)

probe_cases = [
    ('CONTROL', 75, {'gki': 90.0, 'pk': 3e-4, 'd': 0.05, 'gt': 2.0, 'gs': 10.0, 'zth': 70.0, 'zs': 2.5, 'eps': 0.002, 'eps_middle': 1.0, 'wo': 1400.0, 'wo_middle': 1.0, 'ca': 500.0, 'gl_a': 5.0, 'Va_l': -70.0, 'Va_s': -92.0, 'switching_function': 'sigmoid', 'w_a': 2000.0}),
    ('MFA', 125, {'gki': 40.0, 'pk': 5e-5, 'd': 1.5, 'gt': 4.0, 'gs': 22.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 2500.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'tanh', 'w_a': 2000.0}),
    ('MFA_BA', 100, {'gki': 25.0, 'pk': 2e-4, 'd': 1.5, 'gt': 7.0, 'gs': 14.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 1700.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'hill', 'hill_coefficient': 3.0, 'K_d': 1.2, 'w_a': 2000.0}),
]
z = np.array([-80.0, 0.5, 0.2, 0.1], dtype=float)
probe_rows = []
for exp_type, current_na, flat in probe_cases:
    pdict = build_paramdict(exp_type, current_na, flat)
    deltas = []
    for t in [0.0, 11173.0, 12000.0, 21140.0, 22000.0]:
        got = model(z, t, pdict)
        ref = reference_model(z, t, pdict)
        deltas.append(float(np.max(np.abs(got - ref))))
    probe_rows.append({'condition': exp_type, 'current_na': current_na, 'max_abs_rhs_delta': max(deltas), 'status': 'exact_within_float_tolerance' if max(deltas) <= 1e-12 else 'mismatch'})
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(OUTPUT_DIR / 'model_alignment_probe.csv', index=False)
display(probe_df)
assert (probe_df['status'] == 'exact_within_float_tolerance').all()

,condition,current_na,max_abs_rhs_delta,status
0,CONTROL,75,2.220446e-16,exact_within_float_tolerance
1,MFA,125,0.000000e+00,exact_within_float_tolerance
2,MFA_BA,100,0.000000e+00,exact_within_float_tolerance


## Step 02 contract carried into Step 04

In [4]:
step02_outputs = load_step02_outputs_or_run(PROJECT_ROOT, reuse_existing=True)
region_counts = step02_outputs['region_condition_cell_counts']
display(region_counts)
contract = acceptance_contract_table()
display(contract)

,region,condition,n_cells,expected_n_cells,matches_expected,small_stratum
0,DH,CONTROL,7,7,True,False
1,DH,MFA,6,6,True,False
2,DH,MFA_BA,6,6,True,False
3,VH,CONTROL,4,4,True,True
4,VH,MFA,7,7,True,False
5,VH,MFA_BA,7,7,True,False


,criterion,scope,operator,value,role
0,trace_rmse_mean_mV,all6,<=,18.0,accepted_by_trace
1,weighted_pass_fraction_mean,all6,>=,0.3,accepted_by_feature_contract
2,heldout_trace_rmse_mV,leave_one_out,<=,20.0,heldout_screen
3,heldout_weighted_pass_fraction,leave_one_out,>=,0.3,heldout_screen
4,ensemble_rank,all6,<=,3.0,accepted_all6_topk
5,holdout_pass_count,cell,>=,3.0,reviewer_facing_cell


## Runtime-safe Step 04 fit run

By default this executed-review cell uses the full 37-cell target scope through `ASTROMODEL_STEP04_SELECTED_FILE_IDS=all`. A comma-separated `ASTROMODEL_STEP04_SELECTED_FILE_IDS` value is reserved for explicit development subsets, not reviewer-facing evidence.

In [5]:
import time
import warnings
import shutil
import sqlite3
import sqlite3

from scipy.integrate import ODEintWarning
from src.atf_io import load_all_cells
from src.effective_candidate_selection import select_effective_diverse_candidates, summarize_effective_diverse_selection
from src.effective_candidate_selection import select_effective_diverse_candidates, summarize_effective_diverse_selection
from src.step04_cell_fits import run_step04_cell_specific_six_sweep_fitting
from src.step04_loss import Step04OptimizerConfig, Step04LossConfig, TraceLossConfig, config_hash, config_to_jsonable
from src.step04_outputs import save_step04_run_snapshot

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ModuleNotFoundError:
    optuna = None

warnings.filterwarnings("ignore", category=ODEintWarning)


def _env_int_or_none(name, default="all"):
    raw = os.environ.get(name, default).strip().lower()
    if raw in {"", "none", "all"}:
        return None
    return int(raw)


def _env_int(name, default):
    return int(os.environ.get(name, str(default)))


def _env_float(name, default):
    return float(os.environ.get(name, str(default)))


def _env_bool(name, default="0"):
    return os.environ.get(name, str(default)).strip().lower() in {"1", "true", "yes"}


def _env_worker_count(name, default):
    raw = os.environ.get(name, str(default)).strip().lower()
    return "auto" if raw in {"", "auto", "all"} else int(raw)


def _select_file_ids_per_group(project_root, per_group):
    if per_group is None:
        return None
    cells = load_all_cells(project_root / "data" / "2_K+ Pumps Data")
    groups = {}
    for cell in cells:
        groups.setdefault((cell.condition, cell.region), []).append(cell.file_id)
    selected = []
    for key in sorted(groups):
        selected.extend(sorted(groups[key])[:per_group])
    return selected


def _resolve_selected_file_ids():
    raw = os.environ.get("ASTROMODEL_STEP04_SELECTED_FILE_IDS", "all").strip()
    if raw.lower() in {"", "none", "all"}:
        return None
    if raw.lower() in {"group_balanced", "per_group", "two_per_group"}:
        return _select_file_ids_per_group(
            PROJECT_ROOT,
            _env_int_or_none("ASTROMODEL_STEP04_CELLS_PER_GROUP", "2"),
        )
    return [x.strip() for x in raw.split(",") if x.strip()]


STEP04_GENERATION_PRESET = "random_acceptance_160_archive"
STEP04_GENERATION_DEFAULTS = {
    "optimizer_backend": "optuna_scalar",
    "optuna_sampler": "random",
    "optuna_objective": "acceptance_margin",
    "optuna_n_trials": "160",
    "candidate_top_k": "160",
    "n_fit_points": "24",
    "n_starts": "4",
    "max_nfev_all6": "20",
    "accepted_top_k_per_cell": "1000",
    "effective_diverse_candidates_per_cell": "5",
}
STEP04_REVIEWER_VALIDATION_DEFAULTS = {
    # The benchmark disabled holdout to isolate generation. The notebook keeps
    # held-out screening enabled because it is part of the reviewer-facing Step 04 contract.
    "run_holdout": "1",
    "max_nfev_holdout": "40",
}


def _effective_diverse_settings():
    return {
        "candidates_per_cell": _env_int(
            "ASTROMODEL_STEP04_EFFECTIVE_DIVERSE_CANDIDATES_PER_CELL",
            STEP04_GENERATION_DEFAULTS["effective_diverse_candidates_per_cell"],
        ),
        "strategy": os.environ.get("ASTROMODEL_STEP04_EFFECTIVE_DIVERSE_SELECTION_STRATEGY", "quality_filtered_effective_maximin"),
        "distance_threshold": _env_float("ASTROMODEL_STEP04_EFFECTIVE_DIVERSE_DISTANCE_THRESHOLD", "0.5"),
    }


def _ensure_effective_diverse_outputs(path, *, candidates_per_cell, strategy, distance_threshold):
    path = Path(path)
    accepted_path = path / "accepted_cell_ensembles.csv"
    if not accepted_path.exists():
        raise FileNotFoundError(f"Cannot derive effective-diverse candidates without {accepted_path}")
    accepted = pd.read_csv(accepted_path)
    effective_diverse = select_effective_diverse_candidates(
        accepted,
        candidates_per_cell=candidates_per_cell,
        strategy=strategy,
        distance_threshold=distance_threshold,
    )
    effective_summary = summarize_effective_diverse_selection(
        effective_diverse,
        distance_threshold=distance_threshold,
    )
    effective_diverse.to_csv(path / "effective_diverse_cell_ensembles.csv", index=False)
    effective_summary.to_csv(path / "effective_diverse_selection_summary.csv", index=False)
    db_path = path / "step04_cell_fits.sqlite"
    if db_path.exists():
        with sqlite3.connect(db_path) as con:
            effective_diverse.to_sql("effective_diverse_cell_ensembles", con, if_exists="replace", index=False)
            effective_summary.to_sql("effective_diverse_selection_summary", con, if_exists="replace", index=False)
    return effective_diverse, effective_summary


def _validate_completed_full_step04_dir(path):
    required = [
        "analysis_summary.json",
        "cell_fit_candidates.csv",
        "accepted_cell_ensembles.csv",
        "heldout_current_screen.csv",
        "cell_fit_quality_summary.csv",
        "acceptance_contract.csv",
        "candidate_sweep_metrics.csv",
        "cell_trace_inventory.csv",
    ]
    missing = [name for name in required if not (path / name).exists()]
    if missing:
        raise FileNotFoundError(f"Completed Step 04 directory is missing required files: {missing}")
    summary = json.loads((path / "analysis_summary.json").read_text(encoding="utf-8"))
    candidates = pd.read_csv(path / "cell_fit_candidates.csv")
    accepted = pd.read_csv(path / "accepted_cell_ensembles.csv")
    heldout = pd.read_csv(path / "heldout_current_screen.csv")
    inventory = pd.read_csv(path / "cell_trace_inventory.csv")
    if int(summary.get("n_cells", 0)) < 37 or inventory["file_id"].nunique() < 37:
        raise ValueError("Completed Step 04 run is not full target scope: expected 37 ATF cells")
    if set(inventory["region"].dropna()) != {"DH", "VH"}:
        raise ValueError("Completed Step 04 run does not contain both DH and VH regions")
    if set(inventory["condition"].dropna()) != {"CONTROL", "MFA", "MFA_BA"}:
        raise ValueError("Completed Step 04 run does not contain all three conditions")
    if candidates.empty or accepted.empty:
        raise ValueError("Completed Step 04 run has no candidate or accepted ensemble rows")
    if heldout["heldout_sweep"].nunique() < 6:
        raise ValueError("Completed Step 04 run does not contain all six held-out sweeps")
    return summary


def _is_project_output_dir(path):
    outputs_root = (PROJECT_ROOT / "outputs").resolve()
    resolved = Path(path).resolve()
    try:
        rel = resolved.relative_to(outputs_root)
    except ValueError:
        return False
    return bool(rel.parts)


def _sync_completed_full_step04_dir(source_dir, target_dir):
    source_dir = Path(source_dir).resolve()
    target_dir = Path(target_dir).resolve()
    if source_dir == target_dir:
        return
    if _is_project_output_dir(target_dir):
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(source_dir, target_dir)
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    for src in source_dir.iterdir():
        if src.is_file():
            shutil.copy2(src, target_dir / src.name)


def _copy_completed_full_step04_run(source_dir, target_dir, notebook_dir, effective_settings):
    source_dir = Path(source_dir).resolve()
    target_dir = Path(target_dir).resolve()
    notebook_dir = Path(notebook_dir).resolve()
    summary = _validate_completed_full_step04_dir(source_dir)
    _sync_completed_full_step04_dir(source_dir, target_dir)
    _sync_completed_full_step04_dir(source_dir, notebook_dir)
    effective_diverse, effective_summary = _ensure_effective_diverse_outputs(target_dir, **effective_settings)
    if notebook_dir != target_dir:
        _ensure_effective_diverse_outputs(notebook_dir, **effective_settings)
    summary.update({
        "notebook_execution_mode": "reused_validated_completed_full_target_run",
        "notebook_reused_source_dir": str(source_dir.relative_to(PROJECT_ROOT) if source_dir.is_relative_to(PROJECT_ROOT) else source_dir),
        "notebook_output_scope": "full_37_cell_target_scope",
        "n_effective_diverse_candidates": int(len(effective_diverse)),
        "effective_diverse_candidates_per_cell": int(effective_settings["candidates_per_cell"]),
        "effective_diverse_selection_strategy": str(effective_settings["strategy"]),
        "effective_diverse_distance_threshold": float(effective_settings["distance_threshold"]),
    })
    (target_dir / "analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (notebook_dir / "analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return {
        "cell_fit_quality_summary": pd.read_csv(target_dir / "cell_fit_quality_summary.csv"),
        "accepted_cell_ensembles": pd.read_csv(target_dir / "accepted_cell_ensembles.csv"),
        "effective_diverse_cell_ensembles": effective_diverse,
        "effective_diverse_selection_summary": effective_summary,
        "heldout_current_screen": pd.read_csv(target_dir / "heldout_current_screen.csv"),
        "cell_fit_candidates": pd.read_csv(target_dir / "cell_fit_candidates.csv"),
        "candidate_sweep_metrics": pd.read_csv(target_dir / "candidate_sweep_metrics.csv"),
        "acceptance_contract": pd.read_csv(target_dir / "acceptance_contract.csv"),
        "cell_trace_inventory": pd.read_csv(target_dir / "cell_trace_inventory.csv"),
    }, summary


selected_file_ids = _resolve_selected_file_ids()
force_rerun = _env_bool("ASTROMODEL_STEP04_FORCE_RERUN", "0")
completed_full_dir = Path(os.environ.get(
    "ASTROMODEL_STEP04_COMPLETED_FULL_RUN_DIR",
    PROJECT_ROOT / "outputs" / "cell_fits",
)).resolve()
effective_diverse_settings = _effective_diverse_settings()

optimizer_config = Step04OptimizerConfig(
    backend=os.environ.get("ASTROMODEL_STEP04_OPTIMIZER_BACKEND", STEP04_GENERATION_DEFAULTS["optimizer_backend"]),
    optuna_n_trials=_env_int("ASTROMODEL_STEP04_OPTUNA_N_TRIALS", STEP04_GENERATION_DEFAULTS["optuna_n_trials"]),
    optuna_sampler=os.environ.get("ASTROMODEL_STEP04_OPTUNA_SAMPLER", STEP04_GENERATION_DEFAULTS["optuna_sampler"]),
    optuna_objective=os.environ.get("ASTROMODEL_STEP04_OPTUNA_OBJECTIVE", STEP04_GENERATION_DEFAULTS["optuna_objective"]),
    hybrid_scipy_pre_nfev=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_PRE_POINTS", "40"),
    hybrid_scipy_post_nfev=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_POST_POINTS", "20"),
    hybrid_refine_top_k=_env_int("ASTROMODEL_STEP04_HYBRID_REFINE_TOP_K", "3"),
    candidate_top_k=_env_int("ASTROMODEL_STEP04_CANDIDATE_TOP_K", STEP04_GENERATION_DEFAULTS["candidate_top_k"]),
    run_holdout=os.environ.get("ASTROMODEL_STEP04_RUN_HOLDOUT", STEP04_REVIEWER_VALIDATION_DEFAULTS["run_holdout"]).lower() in {"1", "true", "yes"},
)
loss_config = Step04LossConfig(
    trace=TraceLossConfig(
        loss_type=os.environ.get("ASTROMODEL_STEP04_TRACE_LOSS_TYPE", "COMBINED"),
        gradient_loss_weight=float(os.environ.get("ASTROMODEL_STEP04_GRADIENT_LOSS_WEIGHT", "20.0")),
        delta_huber=float(os.environ.get("ASTROMODEL_STEP04_DELTA_HUBER", "1.0")),
    ),
    feature_set=os.environ.get("ASTROMODEL_STEP04_FEATURE_SET", "primary_no_redundant"),
    trace_weight=float(os.environ.get("ASTROMODEL_STEP04_TRACE_WEIGHT", "1.0")),
    feature_weight=float(os.environ.get("ASTROMODEL_STEP04_FEATURE_WEIGHT", "1.0")),
    binary_weight=float(os.environ.get("ASTROMODEL_STEP04_BINARY_WEIGHT", "1.0")),
)
expected_optimization_config_hash = config_hash({
    "loss_config": config_to_jsonable(loss_config),
    "optimizer_config": config_to_jsonable(optimizer_config),
})


def _completed_run_matches_current_config(path, expected_hash):
    optimization_path = Path(path) / "optimization_config.json"
    if not optimization_path.exists():
        return False
    payload = json.loads(optimization_path.read_text(encoding="utf-8"))
    return payload.get("optimization_config_hash") == expected_hash


reuse_completed_full_run = (
    (not force_rerun)
    and selected_file_ids is None
    and completed_full_dir.exists()
    and _completed_run_matches_current_config(completed_full_dir, expected_optimization_config_hash)
)

run_started = time.perf_counter()
if reuse_completed_full_run:
    results, step04_summary = _copy_completed_full_step04_run(
        completed_full_dir,
        STEP04_OUTPUT_DIR,
        OUTPUT_DIR,
        effective_diverse_settings,
    )
    snapshot_path = "reused_validated_completed_full_target_run"
else:
    results = run_step04_cell_specific_six_sweep_fitting(
        PROJECT_ROOT,
        output_dir=STEP04_OUTPUT_DIR,
        selected_file_ids=selected_file_ids,
        max_cells=_env_int_or_none("ASTROMODEL_STEP04_MAX_CELLS", "all"),
        n_fit_points=_env_int("ASTROMODEL_STEP04_N_FIT_POINTS", STEP04_GENERATION_DEFAULTS["n_fit_points"]),
        n_starts=_env_int("ASTROMODEL_STEP04_N_STARTS", STEP04_GENERATION_DEFAULTS["n_starts"]),
        n_fit_scipy_pre_points=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_PRE_POINTS", "40"),
        n_fit_optuna_points=_env_int("ASTROMODEL_STEP04_N_FIT_OPTUNA_POINTS", os.environ.get("ASTROMODEL_STEP04_OPTUNA_N_TRIALS", STEP04_GENERATION_DEFAULTS["optuna_n_trials"])),
        n_fit_scipy_post_points=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_POST_POINTS", "20"),
        max_nfunc_ev_all6=_env_int("ASTROMODEL_STEP04_MAX_NFUNC_EV_ALL6", os.environ.get("ASTROMODEL_STEP04_MAX_NFEV_ALL6", STEP04_GENERATION_DEFAULTS["max_nfev_all6"])),
        max_nfunc_ev_holdout=_env_int("ASTROMODEL_STEP04_MAX_NFUNC_EV_HOLDOUT", os.environ.get("ASTROMODEL_STEP04_MAX_NFEV_HOLDOUT", STEP04_REVIEWER_VALIDATION_DEFAULTS["max_nfev_holdout"])),
        accepted_top_k_per_cell=_env_int("ASTROMODEL_STEP04_ACCEPTED_TOP_K_PER_CELL", STEP04_GENERATION_DEFAULTS["accepted_top_k_per_cell"]),
        effective_diverse_candidates_per_cell=effective_diverse_settings["candidates_per_cell"],
        effective_diverse_selection_strategy=effective_diverse_settings["strategy"],
        effective_diverse_distance_threshold=effective_diverse_settings["distance_threshold"],
        loss_config=loss_config,
        optimizer_config=optimizer_config,
        cell_fit_workers=_env_worker_count("ASTROMODEL_STEP04_CELL_WORKERS", "auto"),
    )
    step04_summary_path = STEP04_OUTPUT_DIR / "analysis_summary.json"
    step04_summary = json.loads(step04_summary_path.read_text(encoding="utf-8")) if step04_summary_path.exists() else {}
    step04_summary.update({
        "notebook_execution_mode": "ran_full_target_or_explicit_user_selected_scope",
        "notebook_selected_file_ids": selected_file_ids,
        "notebook_output_scope": "full_37_cell_target_scope" if selected_file_ids is None else "explicit_user_selected_scope",
        "notebook_generation_preset": STEP04_GENERATION_PRESET,
        "notebook_expected_optimization_config_hash": expected_optimization_config_hash,
        "notebook_reviewer_holdout_enabled": bool(optimizer_config.run_holdout),
    })
    (STEP04_OUTPUT_DIR / "analysis_summary.json").write_text(json.dumps(step04_summary, indent=2), encoding="utf-8")
    snapshot_path = save_step04_run_snapshot(
        STEP04_OUTPUT_DIR,
        backup_dir=os.environ.get("ASTROMODEL_STEP04_BACKUP_DIR"),
        label=os.environ.get("ASTROMODEL_STEP04_RUN_LABEL"),
    )

run_elapsed_s = time.perf_counter() - run_started
summary = results["cell_fit_quality_summary"]
accepted = results["accepted_cell_ensembles"]
effective_diverse = results["effective_diverse_cell_ensembles"]
effective_diverse_summary = results["effective_diverse_selection_summary"]
heldout = results["heldout_current_screen"]
candidates = results["cell_fit_candidates"]
contract = results["acceptance_contract"]
inventory = results["cell_trace_inventory"]
for name, frame in {
    "cell_fit_quality_summary.csv": summary,
    "accepted_cell_ensembles.csv": accepted,
    "effective_diverse_cell_ensembles.csv": effective_diverse,
    "effective_diverse_selection_summary.csv": effective_diverse_summary,
    "heldout_current_screen.csv": heldout,
    "cell_fit_candidates.csv": candidates,
    "acceptance_contract.csv": contract,
    "cell_trace_inventory.csv": inventory,
}.items():
    frame.to_csv(OUTPUT_DIR / name, index=False)
step04_summary.update({
    "notebook_elapsed_s": float(run_elapsed_s),
    "notebook_selected_file_ids": selected_file_ids,
    "n_effective_diverse_candidates": int(len(effective_diverse)),
    "effective_diverse_candidates_per_cell": int(effective_diverse_settings["candidates_per_cell"]),
    "effective_diverse_selection_strategy": str(effective_diverse_settings["strategy"]),
    "effective_diverse_distance_threshold": float(effective_diverse_settings["distance_threshold"]),
})
(OUTPUT_DIR / "analysis_summary.json").write_text(json.dumps(step04_summary, indent=2), encoding="utf-8")
print(f"Step 04 elapsed: {run_elapsed_s:.2f} s")
print(f"Execution mode: {step04_summary.get('notebook_execution_mode')}")
print(f"Cells in target inventory: {inventory['file_id'].nunique() if not inventory.empty else 0}")
print(f"Candidates retained in full history: {len(candidates)}")
print(f"Accepted candidates: {len(accepted)}")
print(f"Effective-diverse downstream candidates: {len(effective_diverse)}")
print(f"Snapshot/status: {snapshot_path}")
print(f"SQLite DB: {STEP04_OUTPUT_DIR / 'step04_cell_fits.sqlite'}")
display(summary)
display(effective_diverse_summary)


/home/xav/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 04 elapsed: 878.28 s
Execution mode: ran_full_target_or_explicit_user_selected_scope
Cells in target inventory: 37
Candidates retained in full history: 5920
Accepted candidates: 2827
Effective-diverse downstream candidates: 158
Snapshot/status: {'step': 'step04', 'snapshot_name': 'random_acceptance_160_archive_full_heldout_20260614_20260614_165722', 'snapshot_created_at_local': '20260614_165722', 'output_dir': '/home/xav/code/astromodel_proving/outputs/cell_fits', 'output_schema_version': 'step04_cell_fits_v2', 'downstream_artifacts': {'candidates': 'cell_fit_candidates.csv', 'accepted_ensembles': 'accepted_cell_ensembles.csv', 'effective_diverse_ensembles': 'effective_diverse_cell_ensembles.csv', 'effective_diverse_selection_summary': 'effective_diverse_selection_summary.csv', 'quality_summary': 'cell_fit_quality_summary.csv', 'heldout_screen': 'heldout_current_screen.csv', 'acceptance_contract': 'acceptance_contract.csv', 'sweep_metrics': 'candidate_sweep_metrics.csv', 'trace_in

,file_id,region,condition,n_candidates,n_accepted_candidates,best_candidate_id,best_trace_rmse_mV,best_weighted_pass_fraction,holdout_pass_count,holdout_mean_rmse_mV,holdout_mean_pass_fraction,cell_reviewer_facing
0,1_DH_1_CONTROL,DH,CONTROL,160,4,1_DH_1_CONTROL__cand_45,13.516104,0.333333,2,13.517364,0.346653,False
1,1_DH_2_CONTROL,DH,CONTROL,160,4,1_DH_2_CONTROL__cand_01,7.783345,0.333333,2,7.782390,0.343075,False
2,2_DH_1_CONTROL,DH,CONTROL,160,4,2_DH_1_CONTROL__cand_01,6.021399,0.333333,2,6.020488,0.341941,False
3,3_DH_1_CONTROL,DH,CONTROL,160,4,3_DH_1_CONTROL__cand_01,8.861673,0.333333,2,8.860592,0.340695,False
4,3_DH_2_CONTROL,DH,CONTROL,160,4,3_DH_2_CONTROL__cand_01,8.621824,0.333333,2,8.620770,0.349073,False
5,DH_1_CONTROL,DH,CONTROL,160,4,DH_1_CONTROL__cand_01,6.933250,0.376879,2,6.933350,0.376879,False
6,DH_2_CONTROL,DH,CONTROL,160,4,DH_2_CONTROL__cand_01,8.160694,0.376879,2,8.161763,0.376880,False
7,1_VH_1_CONTROL,VH,CONTROL,160,0,1_VH_1_CONTROL__cand_34,15.323760,0.285714,0,14.238455,0.285714,False
8,1_VH_2_CONTROL,VH,CONTROL,160,0,1_VH_2_CONTROL__cand_34,15.663759,0.285714,0,15.251971,0.285714,False
9,2_VH_1_CONTROL,VH,CONTROL,160,0,2_VH_1_CONTROL__cand_92,16.493881,0.285714,0,16.256338,0.285714,False


,file_id,region,condition,effective_selection_strategy,n_selected,min_pairwise_effective_log_distance,effective_cluster_count,mean_trace_rmse_mV,mean_weighted_pass_fraction,rank1_retained
0,1_DH_1_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,10.007836,0.315688,True
1,1_DH_2_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,10.383904,0.315688,True
2,2_DH_1_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,8.858564,0.315688,True
3,3_DH_1_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,11.411004,0.315688,True
4,3_DH_2_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,11.153397,0.315688,True
5,DH_1_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,9.704796,0.326597,True
6,DH_1_MFA,DH,MFA,quality_filtered_effective_maximin,5,4.872997,5,8.573076,0.364901,False
7,DH_1_MFA_BA,DH,MFA_BA,quality_filtered_effective_maximin,5,4.503975,5,8.165226,0.425479,False
8,DH_2_CONTROL,DH,CONTROL,quality_filtered_effective_maximin,4,1.585121,4,10.719894,0.326597,True
9,DH_2_MFA,DH,MFA,quality_filtered_effective_maximin,5,4.872997,5,7.549084,0.364901,False


## Accepted candidates, held-out screen, and cross-condition audit context

In [6]:
display(contract)

role_explanations = pd.DataFrame([
    {
        "role": "accepted_by_trace",
        "meaning": "candidate mean six-sweep trace RMSE is within the reviewer-facing trace tolerance",
        "impact": "filters out voltage traces that do not match the observed Vm trajectory enough to be mechanistically considered",
    },
    {
        "role": "accepted_by_feature_contract",
        "meaning": "candidate passes enough Step 02 reliability-weighted Vm feature contracts",
        "impact": "prevents trace-only fits from ignoring empirical feature uncertainty and redundancy controls",
    },
    {
        "role": "heldout_screen",
        "meaning": "leave-one-current-out refits predict the held-out sweep within trace and feature thresholds",
        "impact": "screens for beyond-fit current generalization rather than only all-six training fit",
    },
    {
        "role": "accepted_all6_topk",
        "meaning": "candidate is within the retained all-six accepted ensemble rank for the cell",
        "impact": "keeps a full auditable accepted ensemble while preventing unbounded downstream tables",
    },
    {
        "role": "reviewer_facing_cell",
        "meaning": "cell has enough held-out current passes to support reviewer-facing downstream analysis",
        "impact": "downgrades cells where fit quality exists but held-out evidence is insufficient",
    },
])
display(role_explanations)

if accepted.empty:
    print("No accepted candidates for this run.")
else:
    display(accepted[["file_id", "condition", "region", "candidate_id", "mean_trace_rmse_mV", "mean_weighted_pass_fraction", "accepted_all6"]].head(30))
if heldout.empty:
    print("Held-out screen is empty or disabled for this run.")
else:
    display(heldout[["file_id", "heldout_sweep", "heldout_trace_rmse_mV", "heldout_weighted_pass_fraction", "heldout_pass"]].head(30))

candidate_history_audit = pd.DataFrame([{
    "candidate_history_rows": int(len(candidates)),
    "accepted_candidate_rows": int(len(accepted)),
    "n_cells_in_inventory": int(inventory["file_id"].nunique()) if not inventory.empty else 0,
    "n_regions": int(inventory["region"].nunique()) if "region" in inventory else 0,
    "n_conditions": int(inventory["condition"].nunique()) if "condition" in inventory else 0,
    "full_candidate_history_persisted": bool(len(candidates) >= len(accepted) and len(candidates) > 0),
    "canonical_candidate_history_path": str((STEP04_OUTPUT_DIR / "cell_fit_candidates.csv").relative_to(PROJECT_ROOT)),
}])
display(candidate_history_audit)


,criterion,scope,operator,value,role
0,trace_rmse_mean_mV,all6,<=,18.0,accepted_by_trace
1,weighted_pass_fraction_mean,all6,>=,0.3,accepted_by_feature_contract
2,heldout_trace_rmse_mV,leave_one_out,<=,20.0,heldout_screen
3,heldout_weighted_pass_fraction,leave_one_out,>=,0.3,heldout_screen
4,ensemble_rank,all6,<=,1000.0,accepted_all6_topk
5,holdout_pass_count,cell,>=,3.0,reviewer_facing_cell


,role,meaning,impact
0,accepted_by_trace,candidate mean six-sweep trace RMSE is within ...,filters out voltage traces that do not match t...
1,accepted_by_feature_contract,candidate passes enough Step 02 reliability-we...,prevents trace-only fits from ignoring empiric...
2,heldout_screen,leave-one-current-out refits predict the held-...,screens for beyond-fit current generalization ...
3,accepted_all6_topk,candidate is within the retained all-six accep...,keeps a full auditable accepted ensemble while...
4,reviewer_facing_cell,cell has enough held-out current passes to sup...,downgrades cells where fit quality exists but ...


,file_id,condition,region,candidate_id,mean_trace_rmse_mV,mean_weighted_pass_fraction,accepted_all6
0,1_DH_1_CONTROL,CONTROL,DH,1_DH_1_CONTROL__cand_45,13.516104,0.333333,True
1,1_DH_1_CONTROL,CONTROL,DH,1_DH_1_CONTROL__cand_77,9.766381,0.317061,True
2,1_DH_1_CONTROL,CONTROL,DH,1_DH_1_CONTROL__cand_87,12.471758,0.310287,True
3,1_DH_1_CONTROL,CONTROL,DH,1_DH_1_CONTROL__cand_49,4.277101,0.302069,True
4,1_DH_2_CONTROL,CONTROL,DH,1_DH_2_CONTROL__cand_01,7.783345,0.333333,True
5,1_DH_2_CONTROL,CONTROL,DH,1_DH_2_CONTROL__cand_123,10.516149,0.317061,True
6,1_DH_2_CONTROL,CONTROL,DH,1_DH_2_CONTROL__cand_106,8.230641,0.310287,True
7,1_DH_2_CONTROL,CONTROL,DH,1_DH_2_CONTROL__cand_141,15.005480,0.302069,True
8,2_DH_1_CONTROL,CONTROL,DH,2_DH_1_CONTROL__cand_01,6.021399,0.333333,True
9,2_DH_1_CONTROL,CONTROL,DH,2_DH_1_CONTROL__cand_123,9.026968,0.317061,True


,file_id,heldout_sweep,heldout_trace_rmse_mV,heldout_weighted_pass_fraction,heldout_pass
0,1_DH_1_CONTROL,1,6.359446,0.571429,True
1,1_DH_1_CONTROL,2,9.490082,0.365631,True
2,1_DH_1_CONTROL,3,12.478384,0.285714,False
3,1_DH_1_CONTROL,4,15.236967,0.285714,False
4,1_DH_1_CONTROL,5,17.650626,0.285714,False
5,1_DH_1_CONTROL,6,19.888677,0.285714,False
6,1_DH_2_CONTROL,1,3.648298,0.571429,True
7,1_DH_2_CONTROL,2,5.459200,0.344164,True
8,1_DH_2_CONTROL,3,7.082676,0.285714,False
9,1_DH_2_CONTROL,4,8.626169,0.285714,False


,candidate_history_rows,accepted_candidate_rows,n_cells_in_inventory,n_regions,n_conditions,full_candidate_history_persisted,canonical_candidate_history_path
0,5920,2827,37,2,3,True,outputs/cell_fits/cell_fit_candidates.csv


## Interpretation boundary

This notebook demonstrates that Step 04 uses the expected reviewer-facing model and produces a full-scope cell-specific accepted ensemble under a six-sweep contract. Full candidate history is persisted in `outputs/cell_fits/cell_fit_candidates.csv` and, when available, `outputs/cell_fits/step04_cell_fits.sqlite`.

What it supports:
- the fitted model is the expected ODE model discussed with reviewers;
- Step 04 uses one shared cell-level mechanism across six sweeps;
- Step 02 region-aware feature contracts and redundancy controls are part of acceptance;
- held-out-sweep screening is part of the reviewer-facing contract;
- full candidate history is auditable, not only accepted rows.

What it does **not** establish by itself:
- biological degeneracy;
- phenotype/pathway claims;
- parameter physiological interpretability.

Those claims require Step 05 mechanism decomposition, Step 06 prediction/perturbation validation, Step 07 assumption sensitivity, Step 08 parameter plausibility, and Step 09 synthesis.

## Post-execution scientific status

Executed status for reviewer response: Step 04 now runs the full 37-cell target scope with the configured `random_acceptance_160_archive` preset (`optuna_scalar`, random sampler, acceptance-margin objective, 160 trials, 160 retained candidates per cell) and held-out screening enabled. The canonical output contains 5920 candidate fits, 2827 accepted candidates, 158 effective-diverse downstream candidates, all six held-out sweeps, and 20 reviewer-facing cells under the current acceptance contract.

Comparison against the pre-rerun hybrid/TPE plus targeted high-budget backup is mixed and scientifically important: all-six accepted candidates increased from 2110 to 2827 and the effective-diverse pool increased from 97 to 158, but reviewer-facing held-out support regressed from 33 to 20 cells. The losses are concentrated in DH CONTROL (7/7 to 0/7 reviewer-facing) and VH MFA (7/7 to 1/7 reviewer-facing), mostly because held-out feature-contract pass fractions fall below threshold despite acceptable trace RMSE. DH MFA, DH MFA_BA, and VH MFA_BA remain fully reviewer-facing.

Conclusion: this run supports R2/R6 auditability by producing a full-scope candidate history and a larger effective-diverse accepted pool, but it does not improve the reviewer-facing Step 04 generalization gate versus the previous targeted run. `random_acceptance_160_archive` should be treated as a fast broad generation stage, not as a sole canonical replacement. A reviewer-mature Step 04 configuration still needs a targeted held-out rescue/refinement stage for cells with holdout pass count < 3 before Steps 05-09 use the candidate pool for biological degeneracy and model-proving claims.
